# Module 04.02 — Saved Searches (`search`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.2 Saved Searches (`search`)

A saved search captures the full state of a **Discover session**: the KQL/Lucene query,
any pinned filters, the selected columns, the sort order, and the surrounding data view.
You can think of it as a reusable "view" into an index — a starting point you can share
with a colleague or embed as a panel inside a dashboard.

The heart of the schema is the `kibanaSavedObjectMeta.searchSourceJSON` field — a JSON
blob stored *inside* the JSON of the saved object. It contains the query, filter array,
and an `indexRefName` string that acts as a pointer name resolved against the `references`
array at load time. This indirection is how Kibana keeps the saved search portable:
the actual Data View ID is stored in `references`, not hard-coded into the query blob.

From a restore perspective, saved searches depend on exactly one Data View. If the
referenced Data View is absent when the search is loaded, Kibana will show an error.
During a feature-state restore this is handled automatically because Data Views are
restored first.

### Create in Kibana UI
1. Go to **Discover**
2. Select the `Kibana Sample Data eCommerce` data view
3. Set a KQL filter: `category : "Men's Clothing"`
4. Add columns: `customer_full_name`, `taxful_total_price`, `order_date`
5. Click **Save** → name it `Men's Clothing Orders`

In [ ]:
heading('4.2 Saved Search — inspect existing (loaded with sample data)')

# The eCommerce sample dataset creates saved searches automatically
searches = find_saved_objects('search')
console.print(f'  Found {len(searches)} saved search(es)')
if searches:
    pp(searches[0], 'search saved object')
    info('Key fields in attributes:')
    info('  title              — display name')
    info('  kibanaSavedObjectMeta.searchSourceJSON — query, filter, index (data view ref)')
    info('  columns            — selected Discover columns')
    info('  sort               — default sort [field, direction]')
    info('references           — array linking to the data view (index-pattern) this search uses')

In [ ]:
# Create a saved search via API
heading('Create saved search via API')

# Get the ecommerce data view id
dv_list = find_saved_objects('index-pattern')
ecom_dv = next((o for o in dv_list if 'ecommerce' in o['attributes'].get('title', '')), None)
if not ecom_dv:
    raise RuntimeError('eCommerce data view not found — re-run 00_setup to reload sample data')
ecom_dv_id = ecom_dv['id']

# Idempotent: reuse existing saved search if already present
existing_searches = find_saved_objects('search')
existing_search = next((o for o in existing_searches if o['attributes'].get('title') == "Men's Clothing Orders"), None)
if existing_search:
    search_id = existing_search['id']
    info(f'Saved search already exists: id={search_id}')
else:
    search_resp = kibana_post('/api/saved_objects/search', {
        'attributes': {
            'title': "Men's Clothing Orders",
            'columns': ['customer_full_name', 'taxful_total_price', 'order_date'],
            'sort': [['order_date', 'desc']],
            'kibanaSavedObjectMeta': {
                'searchSourceJSON': json.dumps({
                    'query': {'query': "category : \"Men's Clothing\"", 'language': 'kuery'},
                    'filter': [],
                    'indexRefName': 'kibanaSavedObjectMeta.searchSourceJSON.index',
                }),
            },
        },
        'references': [{'name': 'kibanaSavedObjectMeta.searchSourceJSON.index', 'type': 'index-pattern', 'id': ecom_dv_id}],
    })
    search_id = search_resp['id']
    success(f"Saved search created: id={search_id}")

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Open Discover to see saved searches and apply the one we just created:
kibana_link('/app/discover', 'Discover — open saved search from the list')

In [ ]:
# Ensure saved search exists before snapshotting (makes cell re-runnable)
existing_searches = find_saved_objects('search')
if not any(o['id'] == search_id for o in existing_searches):
    warn('Saved search missing — recreating before snapshot')
    ecom_dv_id = next(o['id'] for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title',''))
    search_resp = kibana_post('/api/saved_objects/search', {
        'attributes': {
            'title': "Men's Clothing Orders",
            'columns': ['customer_full_name', 'taxful_total_price', 'order_date'],
            'sort': [['order_date', 'desc']],
            'kibanaSavedObjectMeta': {
                'searchSourceJSON': json.dumps({
                    'query': {'query': "category : \"Men's Clothing\"", 'language': 'kuery'},
                    'filter': [],
                    'indexRefName': 'kibanaSavedObjectMeta.searchSourceJSON.index',
                }),
            },
        },
        'references': [{'name': 'kibanaSavedObjectMeta.searchSourceJSON.index', 'type': 'index-pattern', 'id': ecom_dv_id}],
    })
    search_id = search_resp['id']
    success(f'Recreated saved search: id={search_id}')

# Snapshot → delete → restore
snap_delete_restore_cycle('snap-search-test', 'Saved Search')

kibana_delete(f'/api/saved_objects/search/{search_id}')
warn(f'Deleted saved search {search_id}')
assert not any(o['id'] == search_id for o in find_saved_objects('search')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-search-test')
time.sleep(3)

restored = find_saved_objects('search')
assert any(o['id'] == search_id for o in restored), 'Saved search should be restored'
success('Saved search restored!')